# Task 4: Style-Conditioned Face-to-Sketch Generation Using a Conditional GAN
**Dataset:** FS2K Facial Sketch Synthesis Dataset (898 train / 160 val / 1,046 test pairs stratified by style 0, 1, 2)
**Architecture:** U-Net Generator with full skip connections at all levels + 70x70 PatchGAN Discriminator with categorical style embedding conditioning
**Loss Function:** $\mathcal{L}_{\text{cGAN}} = \mathcal{L}_{\text{adv}}(G, D) + \lambda_{\text{L1}} \mathcal{L}_{\text{L1}}(G)$
**Optimization:** 15-Trial Optuna Study tuning $\text{lr}_g, \text{lr}_d, \lambda_{\text{L1}}$, base channels, emb dim, and dropout rate.

### Step 1: Environment Diagnostics & GPU Verification

In [ ]:
import torch
import os
import sys
import subprocess

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. Switch to GPU runtime for reasonable training speeds!")

### Step 2: Mount Google Drive & Clone/Pull GitHub Repository

In [ ]:
# Mount Drive for data and checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/genai-restoration-studio'
PROJECT_DIR = '/content/drive/MyDrive/GenAI-A1'

# Clone or pull repository
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/UsmanBari/genai-restoration-studio.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Install requirements
!pip install -q onnx onnxruntime optuna mlflow

### Step 3: Fast Local NVMe Caching & Dataset Preparation (FS2K)

In [ ]:
import shutil
import json
from data.fs2k import FS2KDataset, get_fs2k_dataloaders

LOCAL_DATA_DIR = '/content/local_data/FS2K'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Candidate zip locations in Drive
drive_zip_candidates = [
    os.path.join(PROJECT_DIR, 'raw/FS2K.zip'),
    os.path.join(PROJECT_DIR, 'raw/FS2K/FS2K.zip'),
    os.path.join(PROJECT_DIR, 'FS2K.zip'),
    os.path.join(PROJECT_DIR, 'data/FS2K.zip')
]

drive_folder_candidates = [
    os.path.join(PROJECT_DIR, 'raw/FS2K/FS2K'),
    os.path.join(PROJECT_DIR, 'raw/FS2K'),
    os.path.join(PROJECT_DIR, 'data/FS2K')
]

# Check if already extracted locally
if not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'photo')):
    zip_found = None
    for zc in drive_zip_candidates:
        if os.path.exists(zc):
            zip_found = zc
            break
    
    if zip_found:
        print(f"Unzipping FS2K dataset from {zip_found} to {LOCAL_DATA_DIR}...")
        !unzip -q -o "{zip_found}" -d /content/local_data/
        if os.path.exists('/content/local_data/FS2K/FS2K'):
            !mv /content/local_data/FS2K/FS2K/* {LOCAL_DATA_DIR}/
    else:
        # Check if extracted folder exists on Drive
        folder_found = None
        for fc in drive_folder_candidates:
            if os.path.exists(os.path.join(fc, 'photo')):
                folder_found = fc
                break
        if folder_found:
            print(f"Copying FS2K folder from {folder_found} to local NVMe {LOCAL_DATA_DIR}...")
            !cp -r "{folder_found}/"* {LOCAL_DATA_DIR}/
        else:
            print("Using direct FS2K root resolution from manifests.")

MANIFEST_DIR = os.path.join(REPO_DIR, 'manifests')
if not os.path.exists(os.path.join(MANIFEST_DIR, 'fs2k_train_manifest.json')):
    MANIFEST_DIR = os.path.join(PROJECT_DIR, 'manifests')

print(f"Manifest Directory: {MANIFEST_DIR}")

# Build DataLoaders
train_loader, val_loader, test_loader = get_fs2k_dataloaders(
    manifest_dir=MANIFEST_DIR,
    fs2k_root=LOCAL_DATA_DIR,
    batch_size=16,
    num_workers=0
)

print(f"Train batches: {len(train_loader)} ({len(train_loader.dataset)} pairs)")
print(f"Val batches:   {len(val_loader)} ({len(val_loader.dataset)} pairs)")
print(f"Test batches:  {len(test_loader)} ({len(test_loader.dataset)} pairs)")

### Step 4: Baseline Model Sanity Check (U-Net Generator & PatchGAN Discriminator)

In [ ]:
from models.cgan import StyleConditionedUNetGenerator, ConditionalPatchGANDiscriminator
from training.losses_cgan import ConditionalGANLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dummy_g = StyleConditionedUNetGenerator().to(device)
dummy_d = ConditionalPatchGANDiscriminator().to(device)
criterion = ConditionalGANLoss(lambda_l1=100.0)

batch = next(iter(train_loader))
p = batch['photo'].to(device)
s = batch['sketch'].to(device)
st = batch['style'].to(device)

fake = dummy_g(p, st)
d_real = dummy_d(p, s, st)
d_fake = dummy_d(p, fake.detach(), st)

loss_d, m_d = criterion.discriminator_loss(d_real, d_fake)
loss_g, m_g = criterion.generator_loss(dummy_d(p, fake, st), fake, s)

print(f"Generator Output Shape:     {fake.shape} (Range: [{fake.min():.2f}, {fake.max():.2f}])")
print(f"Discriminator Patch Logits: {d_real.shape}")
print(f"Loss D: {loss_d.item():.4f} (Real Acc: {m_d['d_acc_real']*100:.1f}%, Fake Acc: {m_d['d_acc_fake']*100:.1f}%)")
print(f"Loss G: {loss_g.item():.4f} (Adv: {m_g['loss_g_adv']:.4f}, L1: {m_g['loss_g_l1']:.4f})")
print("[SUCCESS] Baseline cGAN sanity check passed.")

### Step 5: 15-Trial Optuna Hyperparameter Optimization Study

In [ ]:
from training.optuna_cgan import run_optuna_cgan_study

study = run_optuna_cgan_study(
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    n_trials=15,
    epochs_per_trial=5,
    study_name="task4_cgan_fs2k_study"
)

best_params = study.best_params
print("Winning Optuna Hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

### Step 6: Full Schedule 40-Epoch Training of Optimal cGAN Model

In [ ]:
from training.trainer_cgan import train_cgan_full

# Instantiate models with optimal Optuna parameters
best_g = StyleConditionedUNetGenerator(
    in_channels=3,
    out_channels=3,
    num_styles=3,
    emb_dim=int(best_params.get('emb_dim', 32)),
    emb_channels=16,
    base_channels=int(best_params.get('base_channels_g', 64)),
    dropout_rate=float(best_params.get('dropout_rate', 0.5))
).to(device)

best_d = ConditionalPatchGANDiscriminator(
    in_channels=3,
    num_styles=3,
    emb_dim=int(best_params.get('emb_dim', 32)),
    emb_channels=16,
    base_channels=64
).to(device)

save_dir = os.path.join(REPO_DIR, 'checkpoints/task4')
os.makedirs(save_dir, exist_ok=True)

training_results = train_cgan_full(
    net_g=best_g,
    net_d=best_d,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    save_dir=save_dir,
    epochs=40,
    lr_g=float(best_params.get('lr_g', 2e-4)),
    lr_d=float(best_params.get('lr_d', 2e-4)),
    lambda_l1=float(best_params.get('lambda_l1', 100.0)),
    experiment_name="Task4_cGAN_FS2K_Full"
)

### Step 7: Style-Stratified Evaluation on Test Split (Styles 0, 1, 2, Overall)

In [ ]:
from evaluation.benchmark_cgan import run_cgan_benchmark

# Load best checkpoint
best_ckpt_path = os.path.join(save_dir, 'best_generator.pth')
if os.path.exists(best_ckpt_path):
    print(f"Loading best checkpoint from {best_ckpt_path}...")
    ckpt = torch.load(best_ckpt_path, map_location=device)
    best_g.load_state_dict(ckpt['generator_state_dict'] if 'generator_state_dict' in ckpt else ckpt)

benchmark_results = run_cgan_benchmark(
    generator=best_g,
    manifest_path=MANIFEST_DIR,
    fs2k_root=LOCAL_DATA_DIR,
    device=str(device),
    output_dir="artifacts/evaluation_task4",
    num_visualizations=12
)

### Step 8: Qualitative Visualization Across All 3 Styles

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

comp_path = 'artifacts/evaluation_task4/figures/test_synthesis_comparisons.png'
if os.path.exists(comp_path):
    plt.figure(figsize=(12, 16))
    img = Image.open(comp_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title("Test Set Syntheses: Photo | Ground Truth Sketch | cGAN Generated", fontsize=14)
    plt.show()

# Style-transfer visualization (one photo across Style 0, Style 1, Style 2)
best_g.eval()
with torch.no_grad():
    sample_b = next(iter(test_loader))
    test_p = sample_b['photo'][:4].to(device)
    
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for row in range(4):
        p_np = ((test_p[row].permute(1, 2, 0).cpu().numpy() * 0.5) + 0.5).clip(0, 1)
        axes[row, 0].imshow(p_np)
        axes[row, 0].set_title("Input Photo", fontsize=9)
        axes[row, 0].axis('off')
        
        for st_idx in range(3):
            st_t = torch.tensor([st_idx], device=device)
            fake_st = best_g(test_p[row:row+1], st_t)
            fake_np = ((fake_st[0].permute(1, 2, 0).cpu().numpy() * 0.5) + 0.5).clip(0, 1)
            axes[row, st_idx + 1].imshow(fake_np)
            axes[row, st_idx + 1].set_title(f"Style {st_idx}", fontsize=9)
            axes[row, st_idx + 1].axis('off')
            
    plt.tight_layout()
    plt.savefig('artifacts/evaluation_task4/figures/multistyle_transfer_matrix.png', dpi=150)
    plt.show()

### Step 9: Generator-Only ONNX Export & Parity Verification

In [ ]:
from models.onnx_export_cgan import export_cgan_generator_to_onnx, verify_cgan_onnx_numerical_equivalence

onnx_path = 'checkpoints/task4/cgan_generator.onnx'
export_cgan_generator_to_onnx(best_g, onnx_path, opset_version=18)

parity_report = verify_cgan_onnx_numerical_equivalence(
    generator=best_g,
    onnx_path=onnx_path,
    atol=1e-4
)

print(f"ONNX Model Size: {parity_report['onnx_size_mb']:.2f} MB")
print(f"Numerical Parity Status: {parity_report['equivalent']}")

### Step 10: Commit Artifacts & Sync Checkpoints to Google Drive

In [ ]:
# Copy checkpoints and evaluation artifacts to Drive
DRIVE_TASK4_DIR = os.path.join(PROJECT_DIR, 'checkpoints/task4')
os.makedirs(DRIVE_TASK4_DIR, exist_ok=True)

!cp -r checkpoints/task4/* "{DRIVE_TASK4_DIR}/"

DRIVE_ARTIFACTS_DIR = os.path.join(PROJECT_DIR, 'artifacts/evaluation_task4')
os.makedirs(DRIVE_ARTIFACTS_DIR, exist_ok=True)
!cp -r artifacts/evaluation_task4/* "{DRIVE_ARTIFACTS_DIR}/"

print(f"[SUCCESS] All Task 4 checkpoints and figures synchronized to {PROJECT_DIR}.")